# FraudGraph — Stage 3: PureSAGE vs HybridGatedGNN ablation (v3)

Two models trained on the same graph, compared head-to-head.

**v3 protocol fixes (this is the version whose numbers we report):**

1. **Proper 70/10/20 temporal split.** Previous runs early-stopped and picked
   checkpoints on *test* AP — selection leakage that inflated GNN numbers and
   biased the comparison vs XGBoost (which had no such benefit). Now: last
   12.5% of the train period is a validation slice (70/10/20 of the timeline).
   All selection (early stop, checkpoint, winner) happens on **val AP**; the
   test set is scored exactly once per model at the end.
2. **Training message passing sees only the train period.** Previously the
   full graph (incl. test-period nodes) was available during training, letting
   train nodes aggregate future features through reverse edges. Training now
   runs on the train-period subgraph. Test inference uses the full graph —
   matching production, where the serving graph contains everything up to
   "now". Test-period cards/devices unseen in training keep their randomly
   initialized embeddings (realistic cold-start).
3. **Focal loss is alpha-only** (alpha=0.9, gamma=2, no pos_weight). The old
   run stacked pos_weight=27.5 with alpha=0.25 — two imbalance corrections
   partially cancelling into an opaque ~9:1. Same effective weighting, one
   mechanism.

Expect the GNN numbers to come in *lower* than the v2 run — that drop is the
leakage we removed, not a regression.

## 0a. Base installs

In [1]:
!pip install -q torch_geometric mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.4

## 0b. Install PyG C++ sampling backends (pyg-lib + torch-sparse)

**Restart the kernel after this cell succeeds**, then Run All from the top.

In [2]:
import torch, subprocess, sys

TORCH_VER = torch.__version__.split("+")[0]
CUDA_VER  = (torch.version.cuda or "").replace(".", "")
maj, minor, patch = TORCH_VER.split(".")
print(f"detected torch={TORCH_VER}  cuda={torch.version.cuda}")
assert CUDA_VER, "no CUDA detected — enable GPU T4 in notebook Settings"

candidates = [f"{maj}.{minor}.{patch}+cu{CUDA_VER}", f"{maj}.{minor}.0+cu{CUDA_VER}"]
ok = False
for slug in candidates:
    url = f"https://data.pyg.org/whl/torch-{slug}.html"
    print(f"\ntrying {url}")
    rc = subprocess.call([sys.executable, "-m", "pip", "install", "-q",
                          "pyg-lib", "torch-sparse", "-f", url])
    if rc == 0:
        print(f"installed pyg-lib + torch-sparse from {url}")
        ok = True
        break
assert ok, f"no matching wheel found for {candidates}"

detected torch=2.10.0  cuda=12.8

trying https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 69.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 92.8 MB/s eta 0:00:00
installed pyg-lib + torch-sparse from https://data.pyg.org/whl/torch-2.10.0+cu128.html


## 0c. Regular imports + backend check

In [3]:
import os, time, pickle
import numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader
from torch_geometric.transforms import ToUndirected
from torch_geometric.nn import HeteroConv, SAGEConv, GATv2Conv
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
import mlflow

try:
    import pyg_lib; print("pyg-lib OK")
except ImportError:
    try:
        import torch_sparse; print("torch-sparse OK")
    except ImportError:
        raise RuntimeError("Neither pyg-lib nor torch-sparse loaded. Restart kernel and rerun 0b.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu")
OUT = "/kaggle/working"

pyg-lib OK
device: cuda | Tesla T4


## 1. Load graph + reverse edges

In [4]:
import glob
hits = glob.glob("/kaggle/input/**/graph.pt", recursive=True)
assert hits, "graph.pt not found anywhere under /kaggle/input"
GRAPH_PATH = hits[0]
print("loading:", GRAPH_PATH)

data = torch.load(GRAPH_PATH, weights_only=False)
data = ToUndirected(merge=False)(data)
print(data)

loading: /kaggle/input/datasets/svkayy/fraudgraph-stage2/graph.pt
HeteroData(
  transaction={
    x=[590540, 437],
    y=[590540],
    time=[590540],
    train_mask=[590540],
    test_mask=[590540],
  },
  card={ num_nodes=13553 },
  merchant={ num_nodes=527 },
  device={ num_nodes=1787 },
  (transaction, uses_card, card)={ edge_index=[2, 590540] },
  (transaction, at_merchant, merchant)={ edge_index=[2, 524834] },
  (transaction, on_device, device)={ edge_index=[2, 118666] },
  (transaction, velocity_cluster, transaction)={ edge_index=[2, 114439] },
  (card, rev_uses_card, transaction)={ edge_index=[2, 590540] },
  (merchant, rev_at_merchant, transaction)={ edge_index=[2, 524834] },
  (device, rev_on_device, transaction)={ edge_index=[2, 118666] },
  (transaction, rev_velocity_cluster, transaction)={ edge_index=[2, 114439] }
)


## 2. Temporal 70/10/20 split + train-period subgraph

Transaction nodes are time-sorted (asserted in Stage 2), so index ranges are
time ranges. Seeds:
- **train** = first 87.5% of the train period (70% of the timeline)
- **val**   = last 12.5% of the train period (10% of the timeline)
- **test**  = the held-out 20%, scored once at the end

Training and validation message passing run on the **train-period subgraph**
(no test-period nodes reachable). Test inference runs on the full graph.

In [5]:
n_tx = data["transaction"].num_nodes
train_period_mask = data["transaction"].train_mask
n_tr = int(train_period_mask.sum())
assert bool(train_period_mask[:n_tr].all()), "train mask is not a time prefix — graph not time-sorted"

n_seed_tr = int(0.875 * n_tr)
train_seeds = torch.arange(0, n_seed_tr)
val_seeds   = torch.arange(n_seed_tr, n_tr)
test_seeds  = data["transaction"].test_mask.nonzero(as_tuple=False).view(-1)
print(f"train seeds: {train_seeds.numel():,}   val seeds: {val_seeds.numel():,}   test seeds: {test_seeds.numel():,}")

sub = data.subgraph({"transaction": torch.arange(n_tr)})
print("\nTrain-period subgraph:")
for et in sub.edge_types:
    print(f"  {et}   {sub[et].edge_index.shape[1]:,} edges (full graph: {data[et].edge_index.shape[1]:,})")

y_train_seeds = sub["transaction"].y[train_seeds]
print(f"\ntrain fraud rate: {y_train_seeds.float().mean():.4f}")
print(f"val fraud rate:   {sub['transaction'].y[val_seeds].float().mean():.4f}")

train seeds: 413,378   val seeds: 59,054   test seeds: 118,108

Train-period subgraph:
  ('transaction', 'uses_card', 'card')   472,432 edges (full graph: 590,540)
  ('transaction', 'at_merchant', 'merchant')   418,671 edges (full graph: 524,834)
  ('transaction', 'on_device', 'device')   99,536 edges (full graph: 118,666)
  ('transaction', 'velocity_cluster', 'transaction')   80,729 edges (full graph: 114,439)
  ('card', 'rev_uses_card', 'transaction')   472,432 edges (full graph: 590,540)
  ('merchant', 'rev_at_merchant', 'transaction')   418,671 edges (full graph: 524,834)
  ('device', 'rev_on_device', 'transaction')   99,536 edges (full graph: 118,666)
  ('transaction', 'rev_velocity_cluster', 'transaction')   80,729 edges (full graph: 114,439)

train fraud rate: 0.0352
val fraud rate:   0.0349


## 3. Model definitions (mirror `src/model.py`)

In [6]:
class HybridGatedConv(nn.Module):
    def __init__(self, in_dim, out_dim, heads=4, dropout=0.2):
        super().__init__()
        per_head = out_dim // heads if heads > 1 else out_dim
        self.sage = SAGEConv((in_dim, in_dim), out_dim, aggr="mean")
        self.gat  = GATv2Conv((in_dim, in_dim), per_head, heads=heads,
                              concat=(heads > 1), add_self_loops=False, dropout=dropout)
        self.gate = nn.Linear(in_dim, 1)
        self.log_temp = nn.Parameter(torch.zeros(1))

    def forward(self, x, edge_index):
        h_sage = self.sage(x, edge_index)
        h_gat  = self.gat(x, edge_index)
        x_dst = x[1] if isinstance(x, tuple) else x
        g = torch.sigmoid(self.gate(x_dst) / self.log_temp.exp())
        return g * h_gat + (1.0 - g) * h_sage

    @torch.no_grad()
    def gate_stats(self, x, edge_index):
        x_dst = x[1] if isinstance(x, tuple) else x
        g = torch.sigmoid(self.gate(x_dst) / self.log_temp.exp())
        return g.mean().item(), g.std().item()

class _Base(nn.Module):
    def __init__(self, in_dim, nc, nm, nd, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.tx_encoder = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.card_emb     = nn.Embedding(nc, hidden_dim)
        self.merchant_emb = nn.Embedding(nm, hidden_dim)
        self.device_emb   = nn.Embedding(nd, hidden_dim)

    def _init(self, x_tx, node_idx):
        h = {"transaction": self.tx_encoder(x_tx)}
        h["card"]     = self.card_emb.weight     if node_idx.get("card")     is None else self.card_emb(node_idx["card"])
        h["merchant"] = self.merchant_emb.weight if node_idx.get("merchant") is None else self.merchant_emb(node_idx["merchant"])
        h["device"]   = self.device_emb.weight   if node_idx.get("device")   is None else self.device_emb(node_idx["device"])
        return h

class PureSAGE(_Base):
    def __init__(self, in_dim, nc, nm, nd, metadata, hidden_dim=128, out_dim=64, dropout=0.2):
        super().__init__(in_dim, nc, nm, nd, hidden_dim, dropout)
        ets = metadata[1]
        self.conv1 = HeteroConv({et: SAGEConv((hidden_dim, hidden_dim), hidden_dim, aggr="mean") for et in ets}, aggr="sum")
        self.conv2 = HeteroConv({et: SAGEConv((hidden_dim, hidden_dim), out_dim, aggr="mean") for et in ets}, aggr="sum")
        self.head  = nn.Sequential(nn.Linear(out_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))

    def encode(self, x_tx, edge_index_dict, node_idx=None):
        h = self._init(x_tx, node_idx or {})
        h = self.conv1(h, edge_index_dict); h = {k: F.relu(v) for k, v in h.items()}
        h = self.conv2(h, edge_index_dict); h = {k: F.relu(v) for k, v in h.items()}
        return h

    def forward(self, x_tx, edge_index_dict, node_idx=None):
        return self.head(self.encode(x_tx, edge_index_dict, node_idx)["transaction"]).squeeze(-1)

class HybridGatedGNN(_Base):
    def __init__(self, in_dim, nc, nm, nd, metadata, hidden_dim=128, out_dim=64, heads=4, dropout=0.2):
        super().__init__(in_dim, nc, nm, nd, hidden_dim, dropout)
        ets = metadata[1]
        self.conv1 = HeteroConv({et: HybridGatedConv(hidden_dim, hidden_dim, heads=heads, dropout=dropout) for et in ets}, aggr="sum")
        self.conv2 = HeteroConv({et: HybridGatedConv(hidden_dim, out_dim,    heads=1,     dropout=dropout) for et in ets}, aggr="sum")
        self.head  = nn.Sequential(nn.Linear(out_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))

    def encode(self, x_tx, edge_index_dict, node_idx=None):
        h = self._init(x_tx, node_idx or {})
        h = self.conv1(h, edge_index_dict); h = {k: F.relu(v) for k, v in h.items()}
        h = self.conv2(h, edge_index_dict); h = {k: F.relu(v) for k, v in h.items()}
        return h

    def forward(self, x_tx, edge_index_dict, node_idx=None):
        return self.head(self.encode(x_tx, edge_index_dict, node_idx)["transaction"]).squeeze(-1)

## 4. Losses

- **weighted BCE** for PureSAGE (the industry-standard configuration).
- **focal loss, alpha-only** for the Hybrid: alpha=0.9 gives a 9:1 static
  positive weighting; gamma=2 adds dynamic down-weighting of easy examples.
  No pos_weight — stacking it with alpha double-corrects imbalance.

In [7]:
pos_weight = float(((y_train_seeds == 0).sum() / (y_train_seeds == 1).sum()))
pos_weight_t = torch.tensor(pos_weight, device=DEVICE)
print("pos_weight:", round(pos_weight, 1))

def weighted_bce(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight_t)

def focal_bce(logits, targets, alpha=0.9, gamma=2.0):
    ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p = torch.sigmoid(logits)
    p_t = p * targets + (1 - p) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    return (alpha_t * (1 - p_t).pow(gamma) * ce).mean()

pos_weight: 27.4


## 5. Loaders

In [8]:
BATCH = 512
NUM_NEIGHBORS = [15, 10]
EVAL_NEIGHBORS = [25, 15]   # wider fan-out at eval: closer to full message passing

train_loader = NeighborLoader(sub, num_neighbors=NUM_NEIGHBORS,
    input_nodes=("transaction", train_seeds), batch_size=BATCH, shuffle=True)
val_loader = NeighborLoader(sub, num_neighbors=EVAL_NEIGHBORS,
    input_nodes=("transaction", val_seeds), batch_size=1024, shuffle=False)
test_loader = NeighborLoader(data, num_neighbors=EVAL_NEIGHBORS,
    input_nodes=("transaction", test_seeds), batch_size=1024, shuffle=False)

print("train batches:", len(train_loader), "  val batches:", len(val_loader),
      "  test batches:", len(test_loader))

in_dim = data["transaction"].x.shape[1]
metadata = data.metadata()
nc, nm, nd = data["card"].num_nodes, data["merchant"].num_nodes, data["device"].num_nodes

train batches: 808   val batches: 58   test batches: 116


## 6. Training helper (selection on val only; test scored once)

In [9]:
def evaluate(model, loader):
    # Best-effort determinism: seeds the torch RNG the sampler draws from.
    # pyg-lib keeps some internal state, so eval-to-eval variance is reduced,
    # not fully eliminated.
    torch.manual_seed(1234)
    model.eval()
    probs, ys = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            node_idx = {t: batch[t].n_id for t in ["card", "merchant", "device"]}
            logit = model(batch["transaction"].x, batch.edge_index_dict, node_idx=node_idx)
            seed = batch["transaction"].batch_size
            probs.append(torch.sigmoid(logit[:seed]).cpu().numpy())
            ys.append(batch["transaction"].y[:seed].cpu().numpy())
    p = np.concatenate(probs); yv = np.concatenate(ys)
    prec, rec, _ = precision_recall_curve(yv, p)
    return {
        "auc": float(roc_auc_score(yv, p)),
        "average_precision": float(average_precision_score(yv, p)),
        "precision_at_10recall": float(prec[np.argmin(np.abs(rec - 0.10))]),
    }

def train_model(model, loss_fn, tag, epochs=50, eval_every=2, patience=8, lr=1e-3):
    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        batch = next(iter(train_loader)).to(DEVICE)
        node_idx = {t: batch[t].n_id for t in ["card", "merchant", "device"]}
        _ = model(batch["transaction"].x, batch.edge_index_dict, node_idx=node_idx)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=2, min_lr=1e-5)

    history, best = [], {"average_precision": 0.0, "epoch": -1}
    best_state = None; bad = 0

    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time()
        total_loss, seen = 0.0, 0
        for batch in train_loader:
            batch = batch.to(DEVICE)
            node_idx = {t: batch[t].n_id for t in ["card", "merchant", "device"]}
            opt.zero_grad()
            logit = model(batch["transaction"].x, batch.edge_index_dict, node_idx=node_idx)
            seed = batch["transaction"].batch_size
            loss = loss_fn(logit[:seed], batch["transaction"].y[:seed].float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item() * seed; seen += seed
        avg_loss = total_loss / seen

        line = f"[{tag}] epoch {epoch:02d}  loss {avg_loss:.4f}  time {time.time()-t0:.1f}s"
        if epoch % eval_every == 0 or epoch == epochs:
            m = evaluate(model, val_loader)
            sched.step(m["average_precision"])
            line += f"   val AUC {m['auc']:.4f}  val AP {m['average_precision']:.4f}  val P@10R {m['precision_at_10recall']:.4f}"
            history.append({"epoch": epoch, "loss": avg_loss, **{f"val_{k}": v for k, v in m.items()}})
            if m["average_precision"] > best["average_precision"] + 1e-4:
                best = {"epoch": epoch, **m}
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad = 0
            else:
                bad += 1
                if bad >= patience:
                    print(line); print(f"[{tag}] early stop @ epoch {epoch}")
                    break
        print(line)

    model.load_state_dict(best_state)
    test_final = evaluate(model, test_loader)   # the ONLY test-set touch
    print(f"\n[{tag}] best val epoch: {best['epoch']}  (val AP {best['average_precision']:.4f})")
    print(f"[{tag}] TEST: {test_final}")
    return model, best_state, best, test_final, history

## 7. Train Model A — PureSAGE (weighted BCE)

In [10]:
torch.manual_seed(0)
model_A = PureSAGE(in_dim, nc, nm, nd, metadata, hidden_dim=128, out_dim=64, dropout=0.2)
model_A, state_A, val_A, test_A, hist_A = train_model(
    model_A, loss_fn=weighted_bce, tag="SAGE", epochs=50, eval_every=2, patience=8)

[SAGE] epoch 01  loss 0.8344  time 18.7s
[SAGE] epoch 02  loss 0.6653  time 17.3s   val AUC 0.8834  val AP 0.4942  val P@10R 0.9364
[SAGE] epoch 03  loss 0.5797  time 17.3s
[SAGE] epoch 04  loss 0.5210  time 17.1s   val AUC 0.8896  val AP 0.5317  val P@10R 0.9671
[SAGE] epoch 05  loss 0.4614  time 17.3s
[SAGE] epoch 06  loss 0.4206  time 17.3s   val AUC 0.8791  val AP 0.5370  val P@10R 0.9904
[SAGE] epoch 07  loss 0.3896  time 17.2s
[SAGE] epoch 08  loss 0.3644  time 17.3s   val AUC 0.8754  val AP 0.5089  val P@10R 0.9324
[SAGE] epoch 09  loss 0.3411  time 17.5s
[SAGE] epoch 10  loss 0.3272  time 17.3s   val AUC 0.8766  val AP 0.5289  val P@10R 0.9364
[SAGE] epoch 11  loss 0.3053  time 17.3s
[SAGE] epoch 12  loss 0.2999  time 17.1s   val AUC 0.8673  val AP 0.5209  val P@10R 0.9493
[SAGE] epoch 13  loss 0.2316  time 17.5s
[SAGE] epoch 14  loss 0.1979  time 17.2s   val AUC 0.8694  val AP 0.5242  val P@10R 0.9493
[SAGE] epoch 15  loss 0.1823  time 17.2s
[SAGE] epoch 16  loss 0.1699  time 

## 8. Train Model B — HybridGatedGNN (focal, alpha-only)

In [11]:
torch.manual_seed(0)
model_B = HybridGatedGNN(in_dim, nc, nm, nd, metadata, hidden_dim=128, out_dim=64, heads=4, dropout=0.2)
model_B, state_B, val_B, test_B, hist_B = train_model(
    model_B, loss_fn=focal_bce, tag="HYBRID", epochs=50, eval_every=2, patience=8)

[HYBRID] epoch 01  loss 0.0122  time 43.8s
[HYBRID] epoch 02  loss 0.0102  time 43.7s   val AUC 0.8864  val AP 0.4942  val P@10R 0.9717
[HYBRID] epoch 03  loss 0.0094  time 44.2s
[HYBRID] epoch 04  loss 0.0089  time 44.1s   val AUC 0.8970  val AP 0.5328  val P@10R 0.9537
[HYBRID] epoch 05  loss 0.0083  time 44.9s
[HYBRID] epoch 06  loss 0.0080  time 44.5s   val AUC 0.9008  val AP 0.5489  val P@10R 0.9671
[HYBRID] epoch 07  loss 0.0075  time 44.5s
[HYBRID] epoch 08  loss 0.0073  time 44.3s   val AUC 0.9027  val AP 0.5430  val P@10R 0.9321
[HYBRID] epoch 09  loss 0.0070  time 44.3s
[HYBRID] epoch 10  loss 0.0069  time 44.2s   val AUC 0.8945  val AP 0.5353  val P@10R 0.9493
[HYBRID] epoch 11  loss 0.0067  time 44.3s
[HYBRID] epoch 12  loss 0.0066  time 44.1s   val AUC 0.8920  val AP 0.5421  val P@10R 0.9581
[HYBRID] epoch 13  loss 0.0055  time 44.6s
[HYBRID] epoch 14  loss 0.0050  time 45.4s   val AUC 0.8790  val AP 0.5182  val P@10R 0.9581
[HYBRID] epoch 15  loss 0.0048  time 46.5s
[HYBR

### Gate statistics (hybrid model only)

In [12]:
model_B.eval()
with torch.no_grad():
    batch = next(iter(val_loader)).to(DEVICE)
    node_idx = {t: batch[t].n_id for t in ["card", "merchant", "device"]}
    h = model_B._init(batch["transaction"].x, node_idx)
    for et, conv in model_B.conv1.convs.items():
        src, _, dst = et
        x_src = h[src]; x_dst = h[dst]
        x_in = (x_src, x_dst) if src != dst else x_src
        m, s = conv.gate_stats(x_in, batch.edge_index_dict[et])
        print(f"  layer1  {et}  gate mean={m:.2f}  std={s:.2f}  temp={conv.log_temp.exp().item():.2f}")

  layer1  ('transaction', 'uses_card', 'card')  gate mean=0.37  std=0.07  temp=0.56
  layer1  ('transaction', 'at_merchant', 'merchant')  gate mean=0.51  std=0.00  temp=1.00
  layer1  ('transaction', 'on_device', 'device')  gate mean=0.48  std=0.02  temp=0.92
  layer1  ('transaction', 'velocity_cluster', 'transaction')  gate mean=0.80  std=0.10  temp=0.91
  layer1  ('card', 'rev_uses_card', 'transaction')  gate mean=0.53  std=0.30  temp=0.71
  layer1  ('merchant', 'rev_at_merchant', 'transaction')  gate mean=0.70  std=0.19  temp=0.79
  layer1  ('device', 'rev_on_device', 'transaction')  gate mean=0.92  std=0.08  temp=0.71
  layer1  ('transaction', 'rev_velocity_cluster', 'transaction')  gate mean=0.83  std=0.11  temp=0.81


## 9. Side-by-side comparison

Winner chosen by **validation** AP — the test column is reporting-only.

In [13]:
print(f"{'model':16s} {'test AUC':>9s} {'test AP':>8s} {'test P@10R':>11s}")
print("-" * 48)
for name, m in [("xgb_lean", {"auc": 0.7689, "average_precision": 0.1391, "precision_at_10recall": 0.2495}),
                ("xgb_full", {"auc": 0.9067, "average_precision": 0.5241, "precision_at_10recall": 0.9464}),
                ("PureSAGE",  test_A),
                ("Hybrid",    test_B)]:
    print(f"{name:16s} {m['auc']:9.4f} {m['average_precision']:8.4f} {m['precision_at_10recall']:11.4f}")

winner_tag = "HYBRID" if val_B["average_precision"] > val_A["average_precision"] else "SAGE"
print(f"\nWinner by VAL AP: {winner_tag}  (val AP — SAGE: {val_A['average_precision']:.4f}, HYBRID: {val_B['average_precision']:.4f})")

model             test AUC  test AP  test P@10R
------------------------------------------------
xgb_lean            0.7689   0.1391      0.2495
xgb_full            0.9067   0.5241      0.9464
PureSAGE            0.8101   0.4076      0.9083
Hybrid              0.8542   0.4572      0.9553

Winner by VAL AP: HYBRID  (val AP — SAGE: 0.5370, HYBRID: 0.5489)


## 10. Log both runs to MLflow

In [14]:
mlflow.set_tracking_uri(f"sqlite:///{OUT}/mlflow.db")
mlflow.set_experiment("fraudgraph")

for name, val_m, test_m, model, hist, loss_name in [
    ("graphsage_pure_v3",  val_A, test_A, model_A, hist_A, "weighted_bce"),
    ("graphsage_hybrid_v3", val_B, test_B, model_B, hist_B, "focal(alpha=0.9,gamma=2)"),
]:
    with mlflow.start_run(run_name=name):
        mlflow.log_params({
            "arch": type(model).__name__,
            "loss": loss_name,
            "protocol": "70/10/20 temporal; val-selected; train-subgraph message passing",
            "hidden_dim": 128, "out_dim": 64, "dropout": 0.2,
            "lr": 1e-3, "weight_decay": 1e-5,
            "batch_size": BATCH, "num_neighbors": str(NUM_NEIGHBORS),
            "eval_neighbors": str(EVAL_NEIGHBORS),
            "best_val_epoch": val_m["epoch"],
        })
        mlflow.log_metrics({
            "auc": test_m["auc"],
            "average_precision": test_m["average_precision"],
            "precision_at_10recall": test_m["precision_at_10recall"],
            "val_auc": val_m["auc"],
            "val_average_precision": val_m["average_precision"],
        })
        for hrow in hist:
            mlflow.log_metric("val_ap_curve", hrow["val_average_precision"], step=hrow["epoch"])
    print("logged:", name)

2026/07/02 09:03:51 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/02 09:03:51 INFO mlflow.store.db.utils: Updating database tables
2026/07/02 09:03:53 INFO mlflow.tracking.fluent: Experiment with name 'fraudgraph' does not exist. Creating a new experiment.


logged: graphsage_pure_v3
logged: graphsage_hybrid_v3


## 11. Precompute node embeddings from the winner (full graph)

The serving graph legitimately contains all history up to "now", so embeddings
are exported from a full-graph forward pass with the winner's weights.

In [15]:
winner_model = model_B if winner_tag == "HYBRID" else model_A
winner_state = state_B  if winner_tag == "HYBRID" else state_A
winner_arch  = type(winner_model).__name__
print(f">>> Exporting embeddings from: {winner_tag} ({winner_arch}) <<<")

winner_model.eval()
with torch.no_grad():
    x_tx = data["transaction"].x.to(DEVICE)
    edge_index_dict = {et: data[et].edge_index.to(DEVICE) for et in data.edge_types}
    h = winner_model.encode(x_tx, edge_index_dict, node_idx=None)
    for k, v in h.items():
        print(f"  {k}: {tuple(v.shape)}")
    embeddings = {k: v.detach().cpu() for k, v in h.items() if k != "transaction"}
    tx_emb_full = h["transaction"].detach().cpu()

>>> Exporting embeddings from: HYBRID (HybridGatedGNN) <<<
  card: (13553, 64)
  merchant: (527, 64)
  device: (1787, 64)
  transaction: (590540, 64)


## 12. Embedding stack — GNN embeddings as XGBoost features

The published high-EV play on IEEE-CIS: instead of ensembling probabilities,
feed the GNN's **learned transaction embeddings** to XGBoost as extra columns,
letting the tree model combine engineered aggregates and graph structure at
the split level.

Protocol stays clean:
- **Train rows** get embeddings from the **train-subgraph** forward pass (no
  test-period features reach XGBoost's training matrix).
- **Test rows** get embeddings from the full-graph pass (production analog).
- Same fixed hyperparameters as the Stage 1 baselines, no early stopping, test
  scored once.

We also fit a **control**: XGBoost on the same scaled 437-column matrix
*without* embeddings. The control isolates the embeddings' contribution —
`xgb_full` from Stage 1 used the raw (unscaled) matrix, so it isn't a clean
comparator for the delta.

In [16]:
import xgboost as xgb

winner_model.eval()
with torch.no_grad():
    h_sub = winner_model.encode(
        sub["transaction"].x.to(DEVICE),
        {et: sub[et].edge_index.to(DEVICE) for et in sub.edge_types})
    emb_train_period = h_sub["transaction"].cpu().numpy()

X_base = data["transaction"].x.numpy()
y_np = data["transaction"].y.numpy()
emb_full_np = tx_emb_full.numpy()

X_tr_base, X_te_base = X_base[:n_tr], X_base[n_tr:]
y_tr_np, y_te_np = y_np[:n_tr], y_np[n_tr:]
X_tr_stack = np.hstack([X_tr_base, emb_train_period]).astype(np.float32)
X_te_stack = np.hstack([X_te_base, emb_full_np[n_tr:]]).astype(np.float32)
print("stack matrix:", X_tr_stack.shape, "->", X_te_stack.shape)

spw = float((y_tr_np == 0).sum() / (y_tr_np == 1).sum())
XGB_PARAMS = dict(n_estimators=500, max_depth=6, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8,
                  scale_pos_weight=spw, eval_metric="auc",
                  tree_method="hist", device=DEVICE, n_jobs=-1)

def xgb_eval(m, X, yv):
    p = m.predict_proba(X)[:, 1]
    prec, rec, _ = precision_recall_curve(yv, p)
    return {
        "auc": float(roc_auc_score(yv, p)),
        "average_precision": float(average_precision_score(yv, p)),
        "precision_at_10recall": float(prec[np.argmin(np.abs(rec - 0.10))]),
    }

xgb_control = xgb.XGBClassifier(**XGB_PARAMS)
xgb_control.fit(X_tr_base, y_tr_np, verbose=False)
test_xgb_control = xgb_eval(xgb_control, X_te_base, y_te_np)
print("control (437 cols):        ", test_xgb_control)

xgb_stack = xgb.XGBClassifier(**XGB_PARAMS)
xgb_stack.fit(X_tr_stack, y_tr_np, verbose=False)
test_xgb_stack = xgb_eval(xgb_stack, X_te_stack, y_te_np)
print("stack   (437 + 64 emb):    ", test_xgb_stack)

imp = xgb_stack.feature_importances_
emb_share = float(imp[X_base.shape[1]:].sum())
print(f"\nembedding columns' share of total feature importance: {emb_share:.1%}")

stack matrix: (472432, 501) -> (118108, 501)


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [09:04:20] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


control (437 cols):         {'auc': 0.9070612015734636, 'average_precision': 0.5249985398075956, 'precision_at_10recall': 0.9463869463869464}
stack   (437 + 64 emb):     {'auc': 0.872113878783006, 'average_precision': 0.5017899593412029, 'precision_at_10recall': 0.9643705463182898}

embedding columns' share of total feature importance: 31.4%


### Final comparison (all models, test set)

In [17]:
print(f"{'model':22s} {'test AUC':>9s} {'test AP':>8s} {'test P@10R':>11s}")
print("-" * 54)
for name, m in [("xgb_lean (S1)",   {"auc": 0.7689, "average_precision": 0.1391, "precision_at_10recall": 0.2495}),
                ("xgb_full (S1)",   {"auc": 0.9067, "average_precision": 0.5241, "precision_at_10recall": 0.9464}),
                ("PureSAGE",        test_A),
                ("Hybrid",          test_B),
                ("xgb control",     test_xgb_control),
                ("xgb + GNN emb",   test_xgb_stack)]:
    print(f"{name:22s} {m['auc']:9.4f} {m['average_precision']:8.4f} {m['precision_at_10recall']:11.4f}")

delta = test_xgb_stack["auc"] - test_xgb_control["auc"]
print(f"\nGNN-embedding contribution (stack - control): {delta:+.4f} AUC")

with mlflow.start_run(run_name="xgb_scaled_control_v3"):
    mlflow.log_params({**{k: v for k, v in XGB_PARAMS.items() if k != "device"}, "features": "scaled 437"})
    mlflow.log_metrics(test_xgb_control)
with mlflow.start_run(run_name="xgb_gnn_stack_v3"):
    mlflow.log_params({**{k: v for k, v in XGB_PARAMS.items() if k != "device"},
                       "features": "scaled 437 + 64 GNN emb", "embeddings_from": winner_tag})
    mlflow.log_metrics(test_xgb_stack)
print("logged: xgb_scaled_control_v3, xgb_gnn_stack_v3")

model                   test AUC  test AP  test P@10R
------------------------------------------------------
xgb_lean (S1)             0.7689   0.1391      0.2495
xgb_full (S1)             0.9067   0.5241      0.9464
PureSAGE                  0.8101   0.4076      0.9083
Hybrid                    0.8542   0.4572      0.9553
xgb control               0.9071   0.5250      0.9464
xgb + GNN emb             0.8721   0.5018      0.9644

GNN-embedding contribution (stack - control): -0.0349 AUC
logged: xgb_scaled_control_v3, xgb_gnn_stack_v3


## 13. Save artifacts

In [18]:
torch.save(state_A, f"{OUT}/graphsage_pure.pt")
torch.save(state_B, f"{OUT}/graphsage_hybrid.pt")
torch.save(embeddings, f"{OUT}/node_embeddings.pt")
xgb_stack.save_model(f"{OUT}/xgb_gnn_stack.json")

model_meta = {
    "protocol": "70/10/20 temporal; val-selected; train-subgraph message passing",
    "winner": winner_tag,
    "winner_arch": winner_arch,
    "embeddings_from": winner_tag,
    "val_pure": val_A,   "test_pure": test_A,
    "val_hybrid": val_B, "test_hybrid": test_B,
    "test_xgb_control": test_xgb_control,
    "test_xgb_stack": test_xgb_stack,
    "in_dim": in_dim,
    "num_cards": nc, "num_merchants": nm, "num_devices": nd,
    "metadata": metadata,
    "hidden_dim": 128, "out_dim": 64, "heads": 4, "dropout": 0.2,
    "history_pure": hist_A, "history_hybrid": hist_B,
}
with open(f"{OUT}/model_meta.pkl", "wb") as f:
    pickle.dump(model_meta, f)

print("Saved: graphsage_pure.pt, graphsage_hybrid.pt, node_embeddings.pt, model_meta.pkl, mlflow.db")
print(f"Winner: {winner_tag} — node_embeddings.pt is from THIS model; Stage 5 must load the matching weights.")

Saved: graphsage_pure.pt, graphsage_hybrid.pt, node_embeddings.pt, model_meta.pkl, mlflow.db
Winner: HYBRID — node_embeddings.pt is from THIS model; Stage 5 must load the matching weights.
